# [방법론 2 · 2026-10-26] 라우터 붕괴 사전 진단 — 30분 계열(자산 수 N=1·5·10·20·44)

사전 등록(실행 전 고정): D1 분리도 = 평균_t std_e R[t,e] / std_t mean_e R[t,·] (학습 구간 블록 기준), D2 예측가치 = 검증 구간에서
'학습 구간에 맞춘 로지스틱 회귀(라우터 입력 → 다음 블록 최고 전문가)'를 따른 블록 평균 수익 − '학습 최고 고정 전문가' 블록 평균 수익.
라우터 입력은 원본 라우터의 가치 이력 입력을 '학습 최고 고정 전문가의 최근 수익'으로 근사한다(라우터 자기 경로 의존성 제거).
가설: D1 낮음 → 붕괴, D1 높음·D2≤0 → 붕괴 안 하지만 노이즈 전환으로 손해, D1 높음·D2>0 → 쓸모 있는 라우팅.
비교용 실측(이미 확인): 30분 N=1·5·20 전 시드 붕괴, N=10 시드 1개 부분 전환, N=44 거의 붕괴 / 4h·1d 붕괴 / 팩터 전문가(로컬) 붕괴 안 함, D1=0.49.
**학습 없음** — 저장된 모델만 불러온다. CPU로 충분.

- **`런타임 > 모두 실행`만 누르면 된다.** 약 30~60분. `crypto_asset_scaling_rt1006`가 끝나 있어야 한다(모델 로드만 함).
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/router_diagnostic/router_diag_30m.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정, rt1006 재학습과 같은 코드)에서 셀을 불러와 실행한다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

def run_cells(path, ids, patch_after):
    cells = fetch_cells(path)
    for i in ids:
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        if not _ip.run_cell(cells[i]).success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after and not _ip.run_cell(PATCH).success:
            raise RuntimeError('경로 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·레짐 분할 함수·v4 환경 클래스·valid 구간 (학습 없음)
run_cells('crypto/notebooks/crypto_hrl_earnhft_colab.ipynb', [2, 4, 6, 8, 9, 11, 12, 44, 46, 50, 72], patch_after=2)
_s25 = fetch_cells('crypto/notebooks/crypto_hrl_earnhft_colab.ipynb')[25]
exec(_s25[_s25.index('def smooth_perf_row'):_s25.index('smooth_results, smooth_curves = {}')], globals())
print('준비 완료')


In [ ]:
# 실험 설정 — 자산 수 N만 바꾸고 나머지(30분 주기, 연속 비중, top-K+게이트, Q-teacher 없음, 임베딩, β, 예산)는 v4와 같다.
from stable_baselines3 import PPO as SB3_PPO, DQN as SB3_DQN   # 캐시 래퍼 없는 원본 클래스

N_GRID       = [1, 5, 10, 20, 44]   # 세션을 나눠 돌릴 때는 여기서 일부만 남긴다(예: [1, 5] / [10, 20, 44])
ROUTER_SEEDS = [42, 7, 123]         # 라우터 붕괴가 시드 우연인지 보기 위해 N마다 3개
ALPHA        = 0.01                 # 배포 정책과 같은 추론 시점 스무딩
SCALE_DIR    = MODEL_DIR / 'asset_scaling'
SCALE_DIR.mkdir(parents=True, exist_ok=True)

# N<44 부분집합: 거래대금 상위 순(universe.csv), 스테이블·금 연동 자산 제외, 작은 N이 큰 N에 포함되는 중첩 구조.
# N=44는 기존 rt1006 유니버스·풀·매칭 라우터(시드 42)를 그대로 써서 논문 수치와 일치시킨다.
VOLUME_ORDER = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'ZECUSDT', 'OPNUSDT', 'USD1USDT', 'XRPUSDT', 'MUBUSDT', 'BNBUSDT', 'WLDUSDT', 'TRXUSDT', 'DEXEUSDT', 'NEARUSDT', 'DOGEUSDT', 'SNDKBUSDT', 'SPCXBUSDT', 'UNIUSDT', 'AAVEUSDT', 'PEPEUSDT', 'SXTUSDT', 'AVAXUSDT', 'ADAUSDT', 'XAUTUSDT', 'TAOUSDT', 'RLUSDUSDT', 'XLMUSDT', 'DODOUSDT', 'KITEUSDT', 'SUIUSDT', 'LINKUSDT', 'UTKUSDT', 'XECUSDT', 'PAXGUSDT', 'PUMPUSDT', 'ARBUSDT', 'LTCUSDT', 'TREEUSDT', 'TONUSDT', 'UUSDT', 'SYNUSDT', 'ENAUSDT', 'XPLUSDT', 'JTOUSDT', 'BCHUSDT', 'VANRYUSDT', 'DCRUSDT', 'POLUSDT', 'TUSDT', 'ALLOUSDT', 'LUMIAUSDT']
PEGGED = {'USD1USDT', 'UUSDT', 'RLUSDUSDT', 'PAXGUSDT', 'XAUTUSDT'}
ALL_SYMBOLS = list(symbols)
RANKED = [s for s in VOLUME_ORDER if s in ALL_SYMBOLS and s not in PEGGED]
RANKED += sorted(s for s in ALL_SYMBOLS if s not in RANKED and s not in PEGGED)

def symbols_for(n):
    return ALL_SYMBOLS if n == len(ALL_SYMBOLS) else RANKED[:n]

for n in N_GRID:
    print(f'N={n:>2d}:', symbols_for(n) if n <= 20 else f'전체 {len(ALL_SYMBOLS)}종목')


In [ ]:
def kwargs_for(syms):
    return dict(symbols=syms, emb_index=emb_index, embs_all=embs_all,
                ret_index=ret_index, emb_dim=EMB_DIM, tc=TC)


def segment_for(syms):
    """원본 셀 12와 같은 Stage A 레짐 분할 — 부분집합의 등가중 NAV 기준."""
    eq = np.array([np.mean([ret_index.get((s, b), 0.0) for s in syms]) for b in train_buckets])
    nav = pd.Series(np.cumprod(1 + eq), index=range(len(train_buckets)))
    return segment_and_label(nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD,
                             lowpass_window=LOWPASS_WINDOW)


def perf(net):
    """smooth_perf_row와 같은 정의의 숫자형 지표."""
    net = np.asarray(net, dtype=np.float64)
    r = np.diff(net) / net[:-1]
    return {'ret': net[-1] / net[0] - 1,
            'sharpe': r.mean() / (r.std() + 1e-8) * np.sqrt(48 * 365),
            'mdd': ((net / np.maximum.accumulate(net)) - 1).min()}


def run_ew(kw, buckets):
    env = CryptoPortfolioEnv(buckets, **kw)
    env.reset()
    done, zero = False, np.zeros(env.n_assets, dtype=np.float32)
    while not done:
        _, _, done, _, _ = env.step(zero)   # softmax(0) = 거래 가능 종목 등가중
    return np.array(env.value_history)


def run_solo(model, kw, buckets):
    """전문가 하나를 라우터 없이 α=ALPHA 스무딩으로 실행 (원본 셀 80과 같은 로직)."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **kw)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - ALPHA) * env.weights + ALPHA * target_w
        obs, _, done, _, _ = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    return np.array(env.value_history), float(np.mean(env.rho_history))


def run_router(router, pool, kw, buckets):
    """매칭 라우터(+풀)를 α=ALPHA 스무딩으로 실행 (원본 셀 66·76과 같은 로직). 블록별 선택도 반환."""
    env = MacroRouterEnvV4(buckets=buckets, pool_models=pool, route_buckets=ROUTE_BUCKETS,
                           macro_window=MACRO_WINDOW, top_k=TOP_K, episode_progress_mode='constant', **kw)
    obs, _ = env.reset()
    inner, done, chosen = env.inner, False, []
    while not done:
        k, _ = router.predict(obs, deterministic=True)
        chosen.append(int(k))
        done_inner = False
        for _ in range(env.route_buckets):
            a, _ = env.pool_models[int(k)].predict(inner._get_obs(), deterministic=True)
            target_w, rho, top_idx = inner.action_to_weights(a)
            w = (1 - ALPHA) * inner.weights + ALPHA * target_w
            _, _, done_inner, _, info = inner.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            env._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return np.array(inner.value_history), chosen


def oracle(solo_navs):
    """사후 최적 라우터: 라우팅 블록(1일)마다 그날 수익이 가장 큰 전문가의 단독 경로 수익을 이어 붙인다.
    전문가 간 전환 비용·비중 이어받기를 무시하므로 '라우팅으로 얻을 수 있는 몫'의 상한이다."""
    rets = np.stack([np.diff(v) / v[:-1] for v in solo_navs])          # (전문가, 스텝)
    out = []
    for s in range(0, rets.shape[1], ROUTE_BUCKETS):
        blk = rets[:, s:s + ROUTE_BUCKETS]
        out.append(blk[np.argmax(np.prod(1 + blk, axis=1))])
    return np.concatenate([[1.0], np.cumprod(1 + np.concatenate(out))])


def daily_corr(solo_navs):
    """전문가 쌍별 일간 수익률 상관의 평균 — 전문가가 서로 다른 행동을 하는지(상보성의 필요조건)."""
    d = [np.diff(v[::ROUTE_BUCKETS]) / v[::ROUTE_BUCKETS][:-1] for v in solo_navs]
    c = np.corrcoef(np.stack(d))
    return float(c[np.triu_indices_from(c, 1)].mean())
print('함수 정의 완료')


In [ ]:
%%time
# N별: 레짐 분할 → 4개 β 전문가 학습(각 10만 스텝) → 매칭 라우터 학습(시드 3개).
# 이미 저장된 모델은 불러오고 건너뛴다 — 끊기면 `모두 실행`을 다시 누르면 된다.
POOL_KW = dict(learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95,
               clip_range=0.2, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5,
               policy_kwargs=dict(net_arch=[512, 256]), verbose=0, device='cpu', seed=42)
ROUTER_KW = dict(learning_rate=1e-3, buffer_size=20_000, learning_starts=500, batch_size=64, gamma=0.95,
                 train_freq=1, target_update_interval=200, exploration_fraction=0.3,
                 exploration_final_eps=0.05, policy_kwargs=dict(net_arch=[128, 64]), verbose=0, device='cpu')

POOLS, ROUTERS, KW = {}, {}, {}
for n in N_GRID:
    syms = symbols_for(n)
    kw = KW[n] = kwargs_for(syms)
    d = SCALE_DIR / f'N{n}'
    d.mkdir(exist_ok=True)
    is_full = n == len(ALL_SYMBOLS)
    seg_n = seg if is_full else segment_for(syms)
    print(f'\n===== N={n} | 레짐 구간 {len(seg_n.chunks)}개 =====')

    pool_dir = MODEL_DIR / 'crypto_hrl_pool_v4' if is_full else d
    POOLS[n] = {}
    for pool_i, beta in enumerate(POOL_BETAS):
        p = pool_dir / f'agent_beta_{beta}.zip'
        if p.exists():
            POOLS[n][beta] = SB3_PPO.load(str(p), device='cpu')
            print(f'  β={beta}: 로드')
            continue
        env = Monitor(ChunkedSparseGatedEnv(all_buckets=train_buckets, chunks=seg_n.chunks, beta=beta,
                                            risk_threshold=RISK_THRESHOLD, rng=np.random.default_rng(2000 + pool_i),
                                            top_k=TOP_K, **kw))
        m = SB3_PPO('MlpPolicy', env, **POOL_KW)
        m.learn(total_timesteps=POOL_TIMESTEPS_EACH, progress_bar=True)
        m.save(str(p))
        POOLS[n][beta] = m
        print(f'  β={beta}: 학습·저장')

    pool = [POOLS[n][b] for b in POOL_BETAS]
    ROUTERS[n] = {}
    for seed in ROUTER_SEEDS:
        p = (MODEL_DIR / 'crypto_hrl_router_v4_matched.zip') if (is_full and seed == 42) else d / f'router_matched_seed{seed}.zip'
        if p.exists():
            ROUTERS[n][seed] = SB3_DQN.load(str(p), device='cpu')
            print(f'  라우터 시드 {seed}: 로드')
            continue
        env = Monitor(MacroRouterEnvV4(buckets=train_buckets, pool_models=pool, route_buckets=ROUTE_BUCKETS,
                                       macro_window=MACRO_WINDOW, top_k=TOP_K, episode_progress_mode='constant', **kw))
        r = SB3_DQN('MlpPolicy', env, seed=seed, **ROUTER_KW)
        r.learn(total_timesteps=ROUTER_TOTAL_TIMESTEPS, progress_bar=True)
        r.save(str(p))
        ROUTERS[n][seed] = r
        print(f'  라우터 시드 {seed}: 학습·저장')
print('\n학습 완료')


In [ ]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def separation(R: np.ndarray) -> float:
    common = R.mean(axis=1)
    return float(R.std(axis=1).mean() / (common.std() + 1e-12))


def predictive_value(R: np.ndarray, X: np.ndarray, tr: np.ndarray, va: np.ndarray, seed: int = 0) -> dict:
    y = R.argmax(axis=1)
    best_fixed = int(R[tr].mean(axis=0).argmax())
    out = {"최고 고정 전문가": best_fixed, "검증 최고 고정 수익/블록": float(R[va, best_fixed].mean())}
    classes = np.unique(y[tr])
    if len(classes) < 2:
        pred = np.full(len(va), classes[0])
    else:
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=2000, C=1.0, random_state=seed).fit(sc.transform(X[tr]), y[tr])
        pred = clf.predict(sc.transform(X[va]))
    majority = np.bincount(y[tr], minlength=R.shape[1]).argmax()
    out.update({
        "분류 정확도": float((pred == y[va]).mean()),
        "최빈 기준 정확도": float((y[va] == majority).mean()),
        "분류기 수익/블록": float(R[va, pred].mean()) if len(va) else np.nan,
        "D2 예측가치": float(R[va, pred].mean() - R[va, best_fixed].mean()),
        "분류기 최빈 선택 비율": float(np.bincount(pred, minlength=R.shape[1]).max() / len(pred)),
    })
    return out


def diagnose(R, X, tr, va) -> dict:
    return {"D1 분리도": separation(R[tr]), **predictive_value(R, X, tr, va)}


def expert_step_returns(pool_list, kw, buckets):
    """라우터 학습 때와 같은 방식(스무딩 없이 목표 비중 그대로)으로 각 전문가를 단독 실행해 스텝별 수익을 얻는다."""
    out = []
    for m in pool_list:
        env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **kw)
        obs, _ = env.reset(); done = False
        while not done:
            a, _ = m.predict(obs, deterministic=True)
            obs, _, done, _, _ = env.step(a)
        v = np.asarray(env.value_history, dtype=np.float64)
        out.append(np.diff(v) / v[:-1])
    n = min(len(r) for r in out)
    return np.stack([r[:n] for r in out], axis=1)          # (스텝, 전문가)


def block_data(step_rets, route, macro, best):
    """블록(라우팅 단위) 수익 R과, 블록 시작 시점의 라우터 입력 X(학습 최고 고정 전문가의 최근 macro스텝 수익 — 가치 이력 입력의 근사)."""
    starts = list(range(0, step_rets.shape[0] - route + 1, route))
    R = np.stack([np.prod(1 + step_rets[s:s + route], axis=0) - 1 for s in starts])
    hist = step_rets[:, best]
    X = np.stack([np.concatenate([np.zeros(max(0, macro - s)), hist[max(0, s - macro):s]]) for s in starts])
    return R, X


def diagnose_pool(name, pool_list, kw, fit_buckets, valid_buckets_, route, macro):
    r_fit = expert_step_returns(pool_list, kw, fit_buckets)
    r_val = expert_step_returns(pool_list, kw, valid_buckets_)
    Rf, _ = block_data(r_fit, route, macro, 0)
    best = int(Rf.mean(axis=0).argmax())
    Rf, Xf = block_data(r_fit, route, macro, best)
    Rv, Xv = block_data(r_val, route, macro, best)
    R = np.vstack([Rf, Rv]); X = np.vstack([Xf, Xv])
    tr, va = np.arange(len(Rf)), np.arange(len(Rf), len(R))
    corr = np.corrcoef(Rf.T); pair_corr = float(corr[np.triu_indices_from(corr, 1)].mean())
    d = {'설정': name, **diagnose(R, X, tr, va), '전문가 블록수익 평균 상관': pair_corr, '학습 블록 수': len(Rf)}
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in d.items()})
    return d


DIAG_DIR = DATA_DIR / 'crypto' / 'router_diagnostic'; DIAG_DIR.mkdir(parents=True, exist_ok=True)
fit_buckets = [b for b in train_buckets if bucket_to_date(b) <= '2025-09-30']   # D2용 학습 부분(검증 Q4와 분리)
rows = []
for n in N_GRID:
    rows.append(diagnose_pool(f'30분 N={n}', [POOLS[n][b] for b in POOL_BETAS], KW[n], fit_buckets, valid_buckets,
                              ROUTE_BUCKETS, MACRO_WINDOW))
pd.DataFrame(rows).to_csv(DIAG_DIR / 'router_diag_30m.csv', index=False, encoding='utf-8-sig')
print('저장 완료: router_diag_30m.csv')
